# Part 14, notebook 6 of 6: reading the loops, the repeat guard, and the shell that needs a room

Three findings from the framework runs, each one demonstrated here.

**The picture.** (1) A person re-reading the same sentence ten times without writing the answer. (2) A supervisor who taps their shoulder after the third repeat. (3) Giving that person a shell to type into: fine in a test kitchen with no door to the street, not fine on your own desk next to your keys.

Parts 1 and 2 use the saved guard runs and a stand-in call sequence; part 3 runs real Docker containers (no model server needed).

In [1]:
import sys, json, os
from pathlib import Path
os.environ.setdefault("AGENTIC_ZERO_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves (as in the post)
os.environ.setdefault("RUST_LOG", "error"); os.environ.setdefault("LANCE_LOG", "error")   # silence LanceDB deprecation chatter
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))                # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
import subprocess

## 1. The repeat guard: two versions of one rule

The guard ends the run when the same tool call has been made three times. The first version required three **in a row**. It never fired, because the real loop alternated between two calls (`limit: 1`, `limit: 2`). The corrected rule counts three **anywhere in the run**.

**STAND-IN:** the call sequence below imitates the pattern seen in a real looping T1 run (the answer was in context after the first search, then the same line was re-read about 11 times, alternating `limit` 1 and 2). The rule functions are the real logic of `ctx_da_guard.py`.

In [2]:
def sigs_loop():
    seq = [("grep", "id"), ("read_file", "header")]
    for i in range(11):
        seq.append(("read_file", f"line limit={1 + i % 2}"))
    return [json.dumps(c) for c in seq]

def fires_consecutive(sigs, n=3):          # v1: n identical calls in a row
    return next((i + 1 for i in range(len(sigs)) if i >= n - 1 and len(set(sigs[i - n + 1:i + 1])) == 1), None)

def fires_anywhere(sigs, n=3):             # v2 (the one used): the latest call has been made n times in the run
    return next((i + 1 for i in range(len(sigs)) if sigs[:i + 1].count(sigs[i]) >= n), None)

s = sigs_loop()
print("calls in the loop:", len(s))
print("rule v1 (3 in a row)   fires at call:", fires_consecutive(s))
print("rule v2 (3 anywhere)   fires at call:", fires_anywhere(s))

calls in the loop: 13
rule v1 (3 in a row)   fires at call: None
rule v2 (3 anywhere)   fires at call: 7


v1 never fires on this pattern; v2 stops it after 7 calls. That is a property of the *rule*. What happened on the real model is below.

## 2. What the guard did on the three real looping runs

Saved results, `out/ctx_da_guard_runs.json` (the corrected rule) beside `out/old_ctx_da_guard_consecutive_rule.json` (v1).

In [3]:
def show(f):
    for r in json.loads((OUT / f).read_text()):
        print(f"  {r['strategy']:9s} status={r['status'][:34]:34s} correct={r['correct']!s:5s} model calls={r['model_calls_all']:3d} prompt tokens={r['total_prompt_tokens']:8,d}")
print("v1, three in a row:"); show("old_ctx_da_guard_consecutive_rule.json")
print("v2, three anywhere:"); show("ctx_da_guard_runs.json")

v1, three in a row:
  da        status=error: GraphRecursionError: Recurs correct=False model calls= 32 prompt tokens= 213,102
  da+chunk  status=error: GraphRecursionError: Recurs correct=False model calls= 32 prompt tokens= 583,995
  da        status=error: GraphRecursionError: Recurs correct=False model calls= 33 prompt tokens= 706,839
v2, three anywhere:
  da        status=no_answer                          correct=False model calls=  7 prompt tokens=  25,270
  da+chunk  status=ok                                 correct=True  model calls=  4 prompt tokens=  13,270
  da        status=error: GraphRecursionError: Recurs correct=False model calls= 33 prompt tokens= 711,047


**A repeat guard saves tokens; it does not fix the loop.** Of three runs: one stopped after 7 calls and 25,270 tokens (instead of burning a window) with no answer; one answered correctly in 4 calls; one still ran into the recursion limit (33 calls, 711,047 tokens). Temperature 0 on this server is not bit-exact, so the one that flipped to correct is not proof the guard helped. Three runs are an illustration, not a measurement.

## 3. ADK's `Execute` is a real shell: run it in a room

ADK 2.13's experimental `EnvironmentToolset` has `Execute`, which runs a command with **my user's rights and a copy of all my environment variables, API keys included**. A security check flagged it before I had run it on the full set, and it was right. I stopped, set the partial results aside (`old_ctx_adk_env_unsandboxed_runs.json`) and re-ran the arm with every command in a throwaway Alpine container. Here is the exact wrapper:

In [4]:
src = (OUT.parent / "ctx_adk_env.py").read_text()
print(src[src.index("class SandboxedEnvironment"):src.index("async def ask")])

class SandboxedEnvironment(LocalEnvironment):
    """ADK's LocalEnvironment runs `Execute` as a plain shell with my user's rights and a copy of my environment variables
    (API keys included). Every command runs in a throwaway Alpine container instead: no network, read-only root, all
    capabilities dropped, memory and process limits, only the archive copy mounted (at /work, writable), no environment
    variables passed. The shell is therefore busybox `sh`, not bash. (bubblewrap was tried first and cannot set up its network
    namespace on this machine. A first, unsandboxed version of this arm was stopped and discarded; its partial runs are
    out/old_ctx_adk_env_unsandboxed_runs.json.)"""

    async def execute(self, command: str, *, timeout: float | None = None):
        argv = ["docker", "run", "--rm", "--network", "none", "--read-only", "--cap-drop", "ALL",
                "--security-opt", "no-new-privileges", "--pids-limit", "64", "--memory", "512m", "-u", "1000:1000",
     

Now the checks, against a real container started with the same flags. First plant a fake secret in *this* process's environment, then see what the container can reach. **REAL:** Docker, the flags, the commands. **STAND-IN:** the secret is a fake value I made up.

In [5]:
import shlex, tempfile
os.environ["FAKE_API_KEY"] = "sk-this-is-a-fake-secret"
work = Path(tempfile.mkdtemp(prefix="sbx_"))
(work / "note.txt").write_text("a file in the mounted copy")
os.chmod(work, 0o777)
def sandbox(cmd):
    argv = ["docker", "run", "--rm", "--network", "none", "--read-only", "--cap-drop", "ALL", "--security-opt", "no-new-privileges",
            "--pids-limit", "64", "--memory", "512m", "-u", "1000:1000", "-v", f"{work}:/work", "-w", "/work", "-e", "HOME=/work",
            "alpine:latest", "sh", "-c", cmd]
    r = subprocess.run(argv, capture_output=True, text=True, timeout=60)
    return (r.stdout + r.stderr).strip().replace("\n", " | ")[:120]
checks = {
  "can it see my API key?":        "env | grep -c FAKE_API_KEY; echo exit=$?",
  "how many env vars at all?":     "env | wc -l",
  "can it reach the network?":     "wget -q -T 3 -O- http://1.1.1.1 2>&1 | head -1; echo exit=$?",
  "can it write to /usr?":         "touch /usr/x 2>&1",
  "can it read /root?":            "ls /root 2>&1",
  "can it read the mounted copy?": "cat /work/note.txt",
  "can it write in /work?":        "echo ok > /work/new.txt && cat /work/new.txt",
  "which shell is this?":          "echo $0; readlink -f /bin/sh",
}
for q, cmd in checks.items():
    print(f"{q:32s} -> {sandbox(cmd)}")
import shutil; shutil.rmtree(work, ignore_errors=True)

can it see my API key?           -> 0 | exit=1


how many env vars at all?        -> 5


can it reach the network?        -> wget: can't connect to remote host (1.1.1.1): Network unreachable | exit=0


can it write to /usr?            -> touch: /usr/x: Read-only file system


can it read /root?               -> ls: can't open '/root': Permission denied


can it read the mounted copy?    -> a file in the mounted copy


can it write in /work?           -> ok


which shell is this?             -> sh | /bin/busybox


The container sees none of my environment, has no network, cannot write outside the working copy, and cannot read `/root`. A side effect: the shell is busybox `sh`, not bash. Whatever framework you use, a model with a shell wants this boundary.

**Not covered:** a container shares the host kernel (this is not a hardened sandbox against a determined attacker), my user is in the `docker` group (root-equivalent), and these are checks I thought of, not an audit. The same pattern, with 15 attacks and a replay, is Part 5 of the SpatialClaw series.

## And a tool that raises can end the whole run

My first ADK wrapper let a `FileNotFoundError` from `read_file` escape and killed a run. The fix is the same as in my own harness: hand errors back to the model as text. The wrapper:

In [6]:
src = (OUT.parent / "ctx_adk.py").read_text()
i = src.index("def as_text_error")
print(src[i:i + 700])

def as_text_error(fn):
    """Like my harness: a bad path or regex goes back to the model as text instead of ending the run
    (the first ADK pass let a FileNotFoundError kill one run; that run was redone)."""
    @functools.wraps(fn)
    def inner(*a, **kw):
        try:
            return fn(*a, **kw)
        except Exception as e:
            return f"error: {e}"
    return inner


def ls(path: str = ".") -> str:
    """List a directory of the archive (exports/, reports/, policy/)."""
    return h.t_ls(path)


def glob(pattern: str) -> str:
    """Find files by name pattern, e.g. reports/test_0001*.md"""
    return h.t_glob(pattern)


def grep(pattern: str, path: str = ".", mode: str = "
